In [ ]:
# Architecture & Attention Engine

In [8]:
import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, cohen_kappa_score, confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Engine running on: {device}")

class SpatialAttention2D(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // 8, 3, padding=1),
            nn.BatchNorm2d(in_channels // 8),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels // 8, 1, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return x * self.conv(x)

class SOTA_OCT_2D_Net(nn.Module):
    def __init__(self, num_grades=4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(3, 2, 1),
            
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True)
        )
        self.attention = SpatialAttention2D(128)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(64, num_grades)
        )

    def forward(self, x):
        f = self.features(x)
        f = self.attention(f)
        p = self.global_pool(f).view(f.size(0), -1)
        return self.classifier(p)

Engine running on: cuda


In [9]:
# Target-Mapped Dataset Class

In [10]:
class OCTSliceDataset(Dataset):
    def __init__(self, dataframe, data_dir, img_size=128, is_training=True):
        self.df = dataframe
        self.data_dir = data_dir
        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.456], std=[0.224])
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = str(row['image'])
        
        # Build path safely (prevents the img/img/ duplication)
        img_path = os.path.join(self.data_dir, img_name)
        if not os.path.exists(img_path):
            # Fallback if the path logic misaligns
            img_path = os.path.join(self.data_dir, "img", img_name.split('/')[-1])
            
        img = Image.open(img_path).convert('L')
        img_tensor = self.transform(img)
        severity = torch.tensor(row['grade'], dtype=torch.long)
        
        return img_tensor, severity

In [11]:
# Metrics & Pipeline Initialization

In [12]:
def evaluate_performance(model, dataloader, device):
    model.eval()
    all_sev_true = []
    all_sev_pred = []
    
    with torch.no_grad():
        for images, severities in dataloader:
            pred_sev = model(images.to(device))
            all_sev_true.append(severities.numpy())
            all_sev_pred.append(torch.argmax(pred_sev, dim=1).cpu().numpy())
            
    all_sev_true = np.concatenate(all_sev_true)
    all_sev_pred = np.concatenate(all_sev_pred)
    
    qwk = cohen_kappa_score(all_sev_true, all_sev_pred, weights='quadratic')
    macro_f1 = f1_score(all_sev_true, all_sev_pred, average='macro')
    
    print(f"   -> QWK: {qwk:.4f} | Macro F1: {macro_f1:.4f}")
    return qwk

df = pd.read_csv("MMRDR/MMRDR-OCT/OCT.csv")
df['image'] = df['image'].astype(str)
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['grade'])

# FIXED DIRECTORY PATH (Removed the trailing 'img')
IMAGE_ROOT_DIR = "MMRDR/MMRDR-OCT"

# Increased batch size to 16 since 2D is much lighter on VRAM!
train_loader = DataLoader(OCTSliceDataset(train_df, IMAGE_ROOT_DIR, is_training=True), batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(OCTSliceDataset(val_df, IMAGE_ROOT_DIR, is_training=False), batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

model = SOTA_OCT_2D_Net(num_grades=4).to(device)
criterion_severity = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

In [13]:
# training

In [ ]:
num_epochs = 20
best_val_loss = float('inf')

print("Igniting 3D Volumetric Classifier...")
for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    
    for volumes, severities in train_loader:
        volumes, severities = volumes.to(device), severities.to(device)
        
        optimizer.zero_grad()
        loss = criterion_severity(model(volumes), severities)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * volumes.size(0)
        
    train_loss /= len(train_loader.dataset)
    
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for volumes, severities in val_loader:
            volumes, severities = volumes.to(device), severities.to(device)
            val_loss += criterion_severity(model(volumes), severities).item() * volumes.size(0)
            
    val_loss /= len(val_loader.dataset)
    scheduler.step(val_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    evaluate_performance(model, val_loader, device)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "SOTA_OCT_3D_Classifier.pth")
        print("  -> 💾 Deployment Checkpoint Saved!")

print("Training Complete. Ready for deployment.")

Igniting 3D Volumetric Classifier...
Epoch 1/20 | Train Loss: 1.2038 | Val Loss: 1.0563
   -> QWK: 0.0000 | Macro F1: 0.2456
  -> 💾 Deployment Checkpoint Saved!
Epoch 2/20 | Train Loss: 0.9642 | Val Loss: 0.8728
   -> QWK: 0.0389 | Macro F1: 0.2744
  -> 💾 Deployment Checkpoint Saved!
Epoch 3/20 | Train Loss: 0.8617 | Val Loss: 0.7827
   -> QWK: 0.4979 | Macro F1: 0.4875
  -> 💾 Deployment Checkpoint Saved!


In [ ]:
import os
import glob
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms

class OCTVolumetricDataset(Dataset):
    def __init__(self, dataframe, data_dir, depth=32, img_size=128, is_training=True):
        """
        Args:
            dataframe (pd.DataFrame): Dataframe containing 'patient_id' (or volume folder name),
                                      biomarker labels ('IRF', 'SRF', 'PED'), and 'severity_grade'.
            data_dir (str): Root directory where patient/volume folders are stored.
            depth (int): Fixed number of B-scan slices to extract per 3D volume.
            img_size (int): Spatial dimensions (Height, Width) to resize each slice.
            is_training (bool): Whether to apply training augmentations.
        """
        self.df = dataframe
        self.data_dir = data_dir
        self.depth = depth
        self.img_size = img_size
        
        # Spatial transform applied to each 2D slice uniformly
        if is_training:
            self.slice_transform = transforms.Compose([
                transforms.Resize((self.img_size, self.img_size)),
                transforms.ToTensor(),
                # Keep it single channel (Grayscale)
                transforms.Normalize(mean=[0.456], std=[0.224])
            ])
        else:
            self.slice_transform = transforms.Compose([
                transforms.Resize((self.img_size, self.img_size)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.456], std=[0.224])
            ])

    def __len__(self):
        return len(self.df)

    def load_volume(self, volume_folder):
        # Locate all 2D slices for this specific volume scan
        folder_path = os.path.join(self.data_dir, str(volume_folder))
        # Support common extensions (.jpg, .jpeg, .png)
        slice_paths = sorted(glob.glob(os.path.join(folder_path, "*.[jJ][pP][gG]")) + 
                             glob.glob(os.path.join(folder_path, "*.[pP][nN][gG]")))
        
        if len(slice_paths) == 0:
            raise FileNotFoundError(f"No B-scan slices found in directory: {folder_path}")
            
        # Downsample or upsample slice indices to exactly match target self.depth
        total_slices = len(slice_paths)
        indices = np.linspace(0, total_slices - 1, self.depth, dtype=int)
        selected_paths = [slice_paths[i] for i in indices]
        
        volume_slices = []
        for path in selected_paths:
            # Read image as grayscale ('L') as seen in your upload
            img = Image.open(path).convert('L')
            img_tensor = self.slice_transform(img) # Shape: (1, H, W)
            volume_slices.append(img_tensor)
            
        # Stack slices along a new dimension to form the volume: (Depth, Height, Width)
        # squeeze(1) removes the extra channel dim from individual 2D images before stacking
        volume_tensor = torch.stack([s.squeeze(0) for s in volume_slices], dim=0)
        return volume_tensor

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        volume_folder = row['patient_id']
        
        # Load stacked 3D Volume: Shape (Depth, Height, Width)
        volume = self.load_volume(volume_folder)
        
        # Add Channel Dimension back for 3D CNN input -> (1, Depth, Height, Width)
        volume = volume.unsqueeze(0) 
        
        # Extract Biomarkers Multi-label targets [IRF, SRF, PED]
        biomarkers = torch.tensor([row['IRF'], row['SRF'], row['PED']], dtype=torch.float32)
        
        # Extract Multi-class Severity target
        severity = torch.tensor(row['severity_grade'], dtype=torch.long)
        
        return volume, biomarkers, severity

In [ ]:
def train_one_epoch(model, dataloader, optimizer, criterion_bio, criterion_sev, device):
    model.train()
    running_loss = 0.0
    running_bio_loss = 0.0
    running_sev_loss = 0.0
    
    for volumes, biomarkers, severities in dataloader:
        # Move tensors to active GPU
        volumes = volumes.to(device)
        biomarkers = biomarkers.to(device)
        severities = severities.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass through our 3D custom architecture
        pred_biomarkers, pred_severity = model(volumes)
        
        # Compute individual losses
        loss_bio = criterion_bio(pred_biomarkers, biomarkers)
        loss_sev = criterion_sev(pred_severity, severities)
        
        # Combined Loss (Balanced weighting)
        total_loss = loss_bio + loss_sev
        
        # Backward Pass & Optimize
        total_loss.backward()
        optimizer.step()
        
        running_loss += total_loss.item() * volumes.size(0)
        running_bio_loss += loss_bio.item() * volumes.size(0)
        running_sev_loss += loss_sev.item() * volumes.size(0)
        
    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_bio = running_bio_loss / len(dataloader.dataset)
    epoch_sev = running_sev_loss / len(dataloader.dataset)
    
    return epoch_loss, epoch_bio, epoch_sev

print("Training loop setup compiled completely.")

In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
import torch
import copy

# 1. Load and Split the Data
# Ensure your CSV is in the same directory or provide the full path
df = pd.read_csv("MMRDR/MMRDR-OCT/OCT.csv")

# Force patient_id to string so it matches folder names cleanly
df['patient_id'] = df['patient_id'].astype(str)

# 80/20 Split - Stratified by severity to keep class balance intact
train_df, val_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['severity_grade']
)

# 2. Define the explicit path to your image folders
IMAGE_ROOT_DIR = "MMRDR/MMRDR-OCT/img"

# 3. Initialize DataLoaders (Batch size 2 to protect VRAM)
train_dataset = OCTVolumetricDataset(train_df, data_dir=IMAGE_ROOT_DIR, is_training=True)
val_dataset   = OCTVolumetricDataset(val_df,   data_dir=IMAGE_ROOT_DIR, is_training=False)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=2, shuffle=False, num_workers=2, pin_memory=True)

# 4. The Validation Function
def validate(model, dataloader, criterion_bio, criterion_sev, device):
    model.eval()
    running_val_loss = 0.0
    
    with torch.no_grad():
        for volumes, biomarkers, severities in dataloader:
            volumes = volumes.to(device)
            biomarkers = biomarkers.to(device)
            severities = severities.to(device)
            
            pred_bio, pred_sev = model(volumes)
            
            loss_bio = criterion_bio(pred_bio, biomarkers)
            loss_sev = criterion_sev(pred_sev, severities)
            total_loss = loss_bio + loss_sev
            
            running_val_loss += total_loss.item() * volumes.size(0)
            
    return running_val_loss / len(dataloader.dataset)

# 5. The Grand Training Loop
num_epochs = 20
best_val_loss = float('inf')

print("Igniting 3D Volumetric Training Sequence...")
for epoch in range(num_epochs):
    # Train
    train_loss, t_bio, t_sev = train_one_epoch(
        model, train_loader, optimizer, criterion_biomarkers, criterion_severity, device
    )
    
    # Validate
    val_loss = validate(model, val_loader, criterion_biomarkers, criterion_severity, device)
    
    # Step the learning rate scheduler
    scheduler.step(val_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} (Bio:{t_bio:.4f}, Sev:{t_sev:.4f}) | Val Loss: {val_loss:.4f}")
    
    # Checkpointing - Save the absolute best weights for future SaaS deployment
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
        }, "SOTA_OCT_3D_best.pth")
        print("  -> 💾 New SOTA Checkpoint Saved!")

print("Pipeline execution finished.")

KeyError: 'patient_id'

In [9]:
import os

# 1. Where is Jupyter currently looking?
print("Current Directory:", os.getcwd())

# 2. Let's see what is actually inside that MMRDR-OCT folder
folder_path = "mmrdr_project_krrish/MMRDR/MMRDR-OCT/"
if os.path.exists(folder_path):
    print(f"\nContents of {folder_path}:")
    print(os.listdir(folder_path))
else:
    print(f"\nFolder not found at relative path: {folder_path}")

Current Directory: /home/user3/mmrdr_project_krrish

Folder not found at relative path: mmrdr_project_krrish/MMRDR/MMRDR-OCT/


In [ ]:
import copy
from torch.utils.data import DataLoader

# 1. Initialize DataLoaders
# (Keep batch_size small! Start with 2 or 4. If it crashes, drop to 1)
train_dataset = OCTVolumetricDataset(train_df, data_dir="path/to/your/images", is_training=True)
val_dataset   = OCTVolumetricDataset(val_df,   data_dir="path/to/your/images", is_training=False)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=2, shuffle=False, num_workers=2, pin_memory=True)

# 2. The Validation Function
def validate(model, dataloader, criterion_bio, criterion_sev, device):
    model.eval()
    running_val_loss = 0.0
    
    with torch.no_grad():
        for volumes, biomarkers, severities in dataloader:
            volumes = volumes.to(device)
            biomarkers = biomarkers.to(device)
            severities = severities.to(device)
            
            pred_bio, pred_sev = model(volumes)
            
            loss_bio = criterion_bio(pred_bio, biomarkers)
            loss_sev = criterion_sev(pred_sev, severities)
            total_loss = loss_bio + loss_sev
            
            running_val_loss += total_loss.item() * volumes.size(0)
            
    return running_val_loss / len(dataloader.dataset)

# 3. The Grand Loop
num_epochs = 20
best_val_loss = float('inf')

print("Igniting 3D Volumetric Training Sequence...")
for epoch in range(num_epochs):
    # Train
    train_loss, t_bio, t_sev = train_one_epoch(
        model, train_loader, optimizer, criterion_biomarkers, criterion_severity, device
    )
    
    # Validate
    val_loss = validate(model, val_loader, criterion_biomarkers, criterion_severity, device)
    
    # Step the learning rate scheduler
    scheduler.step(val_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} (Bio:{t_bio:.4f}, Sev:{t_sev:.4f}) | Val Loss: {val_loss:.4f}")
    
    # Checkpointing - Save the absolute best version
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
        }, "SOTA_OCT_3D_best.pth")
        print("  -> 💾 New SOTA Checkpoint Saved!")

print("Pipeline execution finished.")

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score, f1_score, cohen_kappa_score, confusion_matrix

def evaluate_performance(model, dataloader, device):
    model.eval()
    
    all_bio_true = []
    all_bio_pred = []
    all_sev_true = []
    all_sev_pred = []
    
    with torch.no_grad():
        for volumes, biomarkers, severities in dataloader:
            volumes = volumes.to(device)
            
            # Forward pass
            pred_bio, pred_sev = model(volumes)
            
            # Convert outputs to probabilities/predictions
            pred_bio_probs = torch.sigmoid(pred_bio).cpu().numpy()
            pred_sev_classes = torch.argmax(pred_sev, dim=1).cpu().numpy()
            
            all_bio_true.append(biomarkers.numpy())
            all_bio_pred.append(pred_bio_probs)
            all_sev_true.append(severities.numpy())
            all_sev_pred.append(pred_sev_classes)
            
    # Stack arrays
    all_bio_true = np.vstack(all_bio_true)
    all_bio_pred = np.vstack(all_bio_pred)
    all_sev_true = np.concatenate(all_sev_true)
    all_sev_pred = np.concatenate(all_sev_pred)
    
    # 1. Evaluate Biomarkers (Multi-label)
    biomarker_names = ['IRF', 'SRF', 'PED']
    print("=== 📊 BIOMARKER PERFORMANCE (MULTI-LABEL) ===")
    for i, name in enumerate(biomarker_names):
        try:
            auc = roc_auc_score(all_bio_true[:, i], all_bio_pred[:, i])
            # Binarize predictions at 0.5 threshold for F1 computation
            f1 = f1_score(all_bio_true[:, i], (all_bio_pred[:, i] > 0.5).astype(int))
            print(f"Target [{name}] -> AUC-ROC: {auc:.4f} | F1-Score: {f1:.4f}")
        except ValueError:
            print(f"Target [{name}] -> Cannot compute AUC (only one class present in batch)")

    # 2. Evaluate Severity Grade (Multi-class)
    print("\n=== 🎯 SEVERITY GRADE PERFORMANCE (MULTI-CLASS) ===")
    qwk = cohen_kappa_score(all_sev_true, all_sev_pred, weights='quadratic')
    macro_f1 = f1_score(all_sev_true, all_sev_pred, average='macro')
    
    print(f"Quadratic Weighted Kappa (QWK): {qwk:.4f} (SOTA Target: >0.85)")
    print(f"Macro F1-Score: {macro_f1:.4f}")
    
    print("\nConfusion Matrix:")
    print(confusion_matrix(all_sev_true, all_sev_pred))

# To run evaluation after an epoch or training finishes:
# evaluate_performance(model, val_loader, device)